# Learning quantum channels on 4×4 sudoku

This notebook trains actual Optyx quantum channels on the recurrent geometry proposed in [optyx#13](https://github.com/rel-int/optyx/issues/13). Sixteen cell channels exchange qubit messages with four row, four column and four square channels while retaining one private memory qubit each. Two to four finite recurrent ticks are contracted with Cotengra on PyTorch tensors, and the Sudoku loss is differentiated through the contraction. There is no classical neural-network interpreter.

The experiment matches the 192-training/64-test record count of [discopy#416](https://github.com/discopy/discopy/pull/416), while making all 256 completed grids distinct and every clue mask uniquely solvable. Three parameter scales are compared under a fixed contraction budget, then only the validation winner is scaled. A sequential GPU ladder first measures the limits of recurrent ticks and compressed bond dimension. This tests the quantum channel, recurrent routing, compressed contraction, and automatic differentiation together. PyTorch is required only for this learning experiment, not by the Optyx core package.

In [1]:
import gc
import os
import time
import warnings

import numpy as np
import torch
from cotengra import ReusableHyperCompressedOptimizer
from discopy import tensor
from quimb.tensor.decomp import register_split_driver

from optyx.channel import Channel, Diagram, qubit
from optyx.core.backends import DiscopyBackend
from optyx.core.contract import contract_tensor
from optyx.core.diagram import Box as CoreBox, bit as core_bit
from optyx.interaction import Box, CMap
from optyx.qubits import Ket

torch.set_num_threads(1)
_ = torch.manual_seed(7)
assert torch.backends.mps.is_available()
assert os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK") == "0"
warnings.filterwarnings(
    "error", message=(
        ".*not currently supported on the MPS backend.*"
        "fall back.*CPU.*"))
device = torch.device("mps")
tensor_dtype = torch.float32
execution_backend = {
    "device": str(device), "dtype": str(tensor_dtype),
    "cpu_fallback": False,
}
execution_backend

{'device': 'mps', 'dtype': 'torch.float32', 'cpu_fallback': False}

## Trainable Optyx channels

Cell maps are shared trainable isometries from seven to nine qubits: six message ports and one private memory qubit enter, then six messages, the updated memory, and two prediction qubits leave. We obtain the isometry by fixing two inputs of a nine-qubit real unitary to fresh zero ancillas. Constraint maps remain eight-qubit unitaries. A conditional rotation partitions the computational basis into pairs that differ at one target qubit and learns a separate real angle for every pair. Cycling the target between layers produces 3,225-, 6,433- and 12,849-parameter models at depths 8, 16 and 32; the largest nearly matches the 12,980 parameters of the MapGNN in discopy#416. Each layer is a vectorised GPU update, and the two local arrays remain shared over every box and tick. The local tensor shapes are unchanged, so unroll depth and compressed bond dimension can be tested independently. The experiment runs in `float32` on PyTorch MPS with CPU fallback disabled. Because MPS does not implement SVD, bond truncation uses a deterministic fixed-rank range projection followed by GPU-native QR; any unsupported operation fails rather than silently moving onto the CPU.

In [2]:
size = 4
n_cells = size ** 2
n_constraints = 3 * size
cell_width = 9
constraint_width = 8


def kron_all(arrays):
    result = arrays[0]
    for array in arrays[1:]:
        result = torch.kron(result, array)
    return result


def rotation_layer(parameters):
    matrices = []
    for theta in parameters:
        matrices.append(torch.stack((
            torch.stack((torch.cos(theta), -torch.sin(theta))),
            torch.stack((torch.sin(theta), torch.cos(theta))),
        )).to(tensor_dtype))
    return kron_all(matrices)


def parameter_count(width, depth):
    return (depth + 1) * width + depth * 2 ** (width - 1)


active_max_bond = 4


def projected_qr(array, max_bond):
    assert array.device.type == "mps"
    rank = min(max_bond, array.shape[-2], array.shape[-1])
    rows = torch.arange(
        array.shape[-1], device=array.device, dtype=array.dtype)[:, None]
    columns = torch.arange(
        rank, device=array.device, dtype=array.dtype)[None, :]
    projection = torch.cos(
        (rows + 1) * (columns + 1) * 0.754877666)
    sketch = array @ projection
    vectors = []
    for column in sketch.mT:
        vector = column
        for previous in vectors:
            vector = vector - previous * torch.dot(previous, vector)
        norm = torch.linalg.vector_norm(vector).clamp_min(1e-7)
        vectors.append(vector / norm)
    left = torch.stack(vectors, dim=1)
    return left, left.mT @ array


@register_split_driver("qr", default_absorb=1)
def bounded_qr(array, absorb=1, max_bond=-1):
    rank = active_max_bond if max_bond < 0 else max_bond
    if absorb in (1, 10, 11):
        left, right = projected_qr(array, rank)
        return (left if absorb != 11 else None, None,
                right if absorb != 10 else None)
    assert absorb in (-1, -10, -11)
    right_t, left_t = projected_qr(array.mT, rank)
    return (left_t.mT if absorb != -11 else None, None,
            right_t.mT if absorb != -10 else None)


@register_split_driver("fixed-rank-qr")
def fixed_rank_qr(array, max_bond=-1, absorb=0):
    assert absorb in (None, -1, 0, 1)
    rank = active_max_bond if max_bond < 0 else max_bond
    left, right = projected_qr(array, rank)
    rank = left.shape[-1]
    singular = None if absorb is not None else torch.ones(
        rank, device=array.device, dtype=array.dtype)
    return left, singular, right


def conditional_rotation(result, parameters, target, basis):
    lower = basis[((basis >> target) & 1) == 0]
    upper = lower ^ (1 << target)
    cosine = torch.cos(parameters)[:, None]
    sine = torch.sin(parameters)[:, None]
    updated = result.clone()
    updated[lower] = (
        cosine * result[lower] - sine * result[upper])
    updated[upper] = (
        sine * result[lower] + cosine * result[upper])
    return updated


def channel_unitary(parameters, width, depth):
    rotations = (depth + 1) * width
    conditional = 2 ** (width - 1)
    basis = torch.arange(2 ** width, device=parameters.device)
    result = rotation_layer(parameters[:width])
    for layer in range(depth):
        offset = rotations + layer * conditional
        result = conditional_rotation(
            result, parameters[offset:offset + conditional],
            layer % width, basis)
        offset = (layer + 1) * width
        result = rotation_layer(
            parameters[offset:offset + width]) @ result
    return result


def trainable_channel(
        name, parameters, dom_width, cod_width, depth):
    unitary = channel_unitary(parameters, cod_width, depth)
    isometry = unitary[:2 ** dom_width]
    kraus = CoreBox(
        name, core_bit ** dom_width, core_bit ** cod_width,
        array=isometry)
    return Channel(
        name, kraus, qubit ** dom_width, qubit ** cod_width)


def initialise_model(
        depth, unroll_steps=1, max_bond=4, seed=7):
    def parameters(width, offset):
        generator = torch.Generator().manual_seed(seed + offset)
        count = parameter_count(width, depth)
        return (torch.randn(
            count, generator=generator, dtype=tensor_dtype
        ).to(device) * .1).requires_grad_()

    return {
        "depth": depth,
        "unroll_steps": unroll_steps,
        "max_bond": max_bond,
        "cell": parameters(cell_width, 0),
        "constraint": parameters(constraint_width, 1),
        "contractions": 0,
    }

## The recurrent sudoku map

A cell reads and writes three message qubits, feeds one private memory qubit back to itself, and writes two prediction qubits to the environment. A constraint reads and writes four messages without private memory or prediction. The two directions give 96 edges and 192 paired feedback wires; the cells add 16 private wires, for 208 memory wires total. Every message port is paired, so the map has an empty boundary and 32 prediction qubits per tick.

In [3]:
def memberships(row, column):
    square = 2 * (row // 2) + column // 2
    position = 2 * (row % 2) + column % 2
    return (
        (n_cells + row, column),
        (n_cells + size + column, row),
        (n_cells + 2 * size + square, position),
    )


def sudoku_cmap(model, channels=None):
    if channels is None:
        cell_channel = trainable_channel(
            "Cell", model["cell"], 7, 9, model["depth"])
        constraint_channel = trainable_channel(
            "Constraint", model["constraint"], 8, 8,
            model["depth"])
    else:
        cell_channel, constraint_channel = channels
    boxes = [
        Box(f"cell_{index}", qubit ** 3, qubit ** 3, cell_channel,
            memory=qubit, prediction=qubit ** 2)
        for index in range(n_cells)
    ]
    boxes += [
        Box(f"{kind}_{index}", qubit ** 4, qubit ** 4,
            constraint_channel)
        for kind in ("row", "column", "square")
        for index in range(size)
    ]
    edges = []
    for cell in range(n_cells):
        row, column = divmod(cell, size)
        for slot, (constraint, position) in enumerate(
                memberships(row, column)):
            edges += [
                ((cell, slot), (constraint, size + position)),
                ((cell, 3 + slot), (constraint, position)),
            ]
    return CMap(boxes, edges)


baseline_model = initialise_model(1)
sudoku = sudoku_cmap(baseline_model)
cell_isometry = sudoku.boxes[0].channel.kraus.array
assert torch.allclose(
    cell_isometry @ cell_isometry.T,
    torch.eye(2 ** 7, dtype=tensor_dtype, device=device), atol=1e-5)
topology = {
    "boxes": len(sudoku.boxes),
    "edges": len(sudoku.edges),
    "boundary_wires": len(sudoku.dom),
    "prediction_wires": len(sudoku.prediction),
    "paired_memory_wires": 2 * len(sudoku.edges),
    "internal_memory_wires": sum(
        len(box.memory) for box in sudoku.boxes),
    "memory_wires": len(sudoku.memory),
    "conditional_parameters": {
        depth: (
            parameter_count(cell_width, depth),
            parameter_count(constraint_width, depth),
        ) for depth in (8, 16, 32)
    },
}
assert topology == {
    "boxes": 28, "edges": 96, "boundary_wires": 0,
    "prediction_wires": 32, "paired_memory_wires": 192,
    "internal_memory_wires": 16, "memory_wires": 208,
    "conditional_parameters": {
        8: (2129, 1096), 16: (4249, 2184),
        32: (8489, 4360)}}
topology

{'boxes': 28,
 'edges': 96,
 'boundary_wires': 0,
 'prediction_wires': 32,
 'paired_memory_wires': 192,
 'internal_memory_wires': 16,
 'memory_wires': 208,
 'ansatz_parameters': {'bipartite': (34, 32),
  'ring': (36, 32),
  'all_pairs': (54, 44)}}

## A larger, leakage-free dataset

We enumerate all 288 completed grids and sample 256 of them without replacement: 192 for training and 64 for testing, matching the record counts of discopy#416. In that notebook the same generator yields only 24 distinct completed grids, and 22 test solution patterns already occur in training. Here every solution is distinct across the split. Each puzzle hides cells 0 and 1 plus six others, and its eight remaining clues uniquely identify one grid in the complete corpus. The negative completion swaps the two hidden values in the first row, so it preserves every clue but violates sudoku constraints.

In [4]:
def sudoku_groups():
    rows = [tuple(row * size + column for column in range(size))
            for row in range(size)]
    columns = [tuple(row * size + column for row in range(size))
               for column in range(size)]
    squares = [
        tuple((2 * block_row + row) * size
              + 2 * block_column + column
              for row in range(2) for column in range(2))
        for block_row in range(2) for block_column in range(2)
    ]
    return rows + columns + squares


groups = sudoku_groups()
peers = [set() for _ in range(n_cells)]
for group in groups:
    for cell in group:
        peers[cell].update(set(group) - {cell})


def enumerate_solutions():
    result, grid = [], [0] * n_cells

    def visit(cell):
        if cell == n_cells:
            result.append(np.array(grid))
            return
        for value in range(1, size + 1):
            if all(grid[peer] != value for peer in peers[cell]):
                grid[cell] = value
                visit(cell + 1)
        grid[cell] = 0

    visit(0)
    return np.stack(result)


solutions = enumerate_solutions()
assert solutions.shape == (288, n_cells)

In [5]:
def puzzle_case(solution, random):
    available = np.arange(2, n_cells)
    while True:
        clue_cells = random.choice(
            available, n_cells // 2, replace=False)
        matches = np.all(
            solutions[:, clue_cells] == solution[clue_cells], axis=1)
        if matches.sum() == 1:
            clue = np.zeros(n_cells, dtype=int)
            clue[clue_cells] = solution[clue_cells]
            invalid = solution.copy()
            invalid[0], invalid[1] = invalid[1], invalid[0]
            return clue, solution, invalid


random = np.random.default_rng(19)
selection = random.choice(len(solutions), 256, replace=False)
cases = [puzzle_case(solutions[index], random) for index in selection]
train_cases, test_cases = cases[:192], cases[192:]
validation_cases = test_cases[:8]
readout_cases = test_cases[:4]
dataset = {
    "complete_corpus": len(solutions),
    "training_puzzles": len(train_cases),
    "held_out_puzzles": len(test_cases),
    "distinct_selected_solutions": len({
        tuple(solution) for _, solution, _ in cases}),
    "test_solutions_seen_in_training": len(
        {tuple(case[1]) for case in train_cases}
        & {tuple(case[1]) for case in test_cases}),
    "clues_per_puzzle": int(np.count_nonzero(train_cases[0][0])),
}
assert dataset["distinct_selected_solutions"] == 256
assert dataset["test_solutions_seen_in_training"] == 0
dataset

{'complete_corpus': 288,
 'training_puzzles': 192,
 'held_out_puzzles': 64,
 'distinct_selected_solutions': 256,
 'test_solutions_seen_in_training': 0,
 'clues_per_puzzle': 8}

## Efficient finite-time tensor semantics

We use the finite semantics of `Diagram.at_time`, with one unrolling (two recurrent ticks), rather than assuming that `fix` converges. Materialising its permutation on 240 qubit wires obscures the useful tensor-network structure, so `at_time_tensor_map` constructs the equivalent pure-Kraus Born-score network directly as a `discopy.tensor.CMap`. At tick zero every paired port starts in `|+⟩` and each private memory starts in `|0⟩`; later ticks connect partner outputs and each cell's memory output to its next input. Clue predictions are postselected on their digit at every tick, free predictions meet uniform effects until the last tick, and the final memory is closed with uniform effects. Thus recurrent routing is encoded by index labels rather than thousands of swap boxes.

In [6]:
zero = torch.tensor(
    [1, 0], dtype=tensor_dtype, device=device)
one = torch.tensor(
    [0, 1], dtype=tensor_dtype, device=device)
plus = torch.tensor(
    [1, 1], dtype=tensor_dtype, device=device) / 2 ** .5


def digit_vectors(values):
    vectors = []
    for value in values:
        if value == 0:
            vectors += [plus, plus]
        else:
            value -= 1
            vectors += [
                one if value // 2 else zero,
                one if value % 2 else zero,
            ]
    return vectors


def at_time_tensor_map(cmap, clues, completion, unroll_steps):
    ticks = unroll_steps + 1
    boxes, pairs, next_port = [], [], 0

    def add(box):
        nonlocal next_port
        base = next_port
        next_port += len(box.dom) + len(box.cod)
        boxes.append(box)
        dom = list(range(base, base + len(box.dom)))
        cod = list(reversed(range(base + len(box.dom), next_port)))
        return dom, cod

    def state(value):
        with tensor.backend("pytorch"):
            box = tensor.Box(
                "state", tensor.Dim(), tensor.Dim(2), value)
        return add(box)[1][0]

    def effect(value):
        with tensor.backend("pytorch"):
            box = tensor.Box(
                "effect", tensor.Dim(2), tensor.Dim(), value)
        return add(box)[0][0]

    assert not cmap.boundary
    paired_inputs = {
        port: state(plus) for port in cmap.paired}
    private_inputs = {
        (box_index, memory_index): state(zero)
        for box_index, box in enumerate(cmap.boxes)
        for memory_index in range(len(box.memory))
    }
    with tensor.backend("pytorch"):
        local_tensors = {
            id(box.channel): tensor.Box(
                box.channel.name,
                tensor.Dim(2) ** len(box.channel.dom),
                tensor.Dim(2) ** len(box.channel.cod),
                box.channel.kraus.array)
            for box in cmap.boxes
        }
    local_inputs, local_outputs, predictions = [], [], []
    for _ in range(ticks):
        step_inputs, step_outputs, step_predictions = {}, {}, {}
        for box_index, box in enumerate(cmap.boxes):
            dom, cod = add(local_tensors[id(box.channel)])
            for port_index in range(len(box.ports)):
                port = box_index, port_index
                step_inputs["port", port] = dom[port_index]
                step_outputs["port", port] = cod[port_index]
            offset = len(box.ports)
            for memory_index in range(len(box.memory)):
                memory = box_index, memory_index
                step_inputs["memory", memory] = (
                    dom[offset + memory_index])
                step_outputs["memory", memory] = (
                    cod[offset + memory_index])
            offset += len(box.memory)
            step_predictions[box_index] = cod[
                offset:offset + len(box.prediction)]
        local_inputs.append(step_inputs)
        local_outputs.append(step_outputs)
        predictions.append(step_predictions)
    for step in range(ticks):
        for port in cmap.paired:
            source = paired_inputs[port] if step == 0 else (
                local_outputs[step - 1][
                    "port", cmap.partner[port]])
            pairs.append((
                source, local_inputs[step]["port", port]))
        for memory, source in private_inputs.items():
            if step:
                source = local_outputs[step - 1]["memory", memory]
            pairs.append((
                source, local_inputs[step]["memory", memory]))
        for cell in range(n_cells):
            value = clues[cell] or (
                completion[cell] if step == ticks - 1 else 0)
            for output, vector in zip(
                    predictions[step][cell], digit_vectors([value])):
                pairs.append((output, effect(vector)))
    for port in cmap.paired:
        pairs.append((
            local_outputs[-1]["port", cmap.partner[port]],
            effect(plus)))
    for memory in private_inputs:
        pairs.append((
            local_outputs[-1]["memory", memory], effect(plus)))
    edges = list(range(next_port))
    for left, right in pairs:
        edges[left], edges[right] = right, left
    assert all(edge != index for index, edge in enumerate(edges))
    return tensor.CMap(
        tensor.Dim(), tensor.Dim(), tuple(boxes), edges)

In [7]:
network_summaries = {}
for ticks in (2, 3, 4):
    network = at_time_tensor_map(
        sudoku, train_cases[0][0], train_cases[0][1], ticks - 1)
    network_summaries[ticks] = {
        "unroll_steps": ticks - 1,
        "tensor_boxes": len(network.boxes),
        "tensor_ports": len(network.ports),
    }
assert network_summaries == {
    2: {"unroll_steps": 1, "tensor_boxes": 536,
        "tensor_ports": 1376},
    3: {"unroll_steps": 2, "tensor_boxes": 596,
        "tensor_ports": 1856},
    4: {"unroll_steps": 3, "tensor_boxes": 656,
        "tensor_ports": 2336},
}
network_summaries

{'unroll_steps': 1, 'ticks': 2, 'tensor_boxes': 536, 'tensor_ports': 1376}

## Born loss, contraction and backpropagation

The scalar contraction is a postselected amplitude for a final prediction. Its squared magnitude is a Born score. We retain the correct-versus-invalid ranking metric, and also obtain a four-way distribution for each hidden cell by postselecting that cell while leaving the other hidden predictions uniform. Cotengra caches one deterministic compressed path per `(ticks, chi)` pair. For training, the two shared local arrays are built once per optimizer update. Each example is differentiated only as far as those arrays and its contraction graph is released; the accumulated array gradients are then propagated through the deep ansatz once. This is the exact mini-batch gradient while keeping one four-contraction graph live at a time.

In [8]:
path_optimizers = {}


def contraction_optimizer(unroll_steps, max_bond):
    key = unroll_steps, max_bond
    if key not in path_optimizers:
        path_optimizers[key] = ReusableHyperCompressedOptimizer(
            chi=max_bond, methods=("greedy-compressed",),
            max_repeats=1, optlib="random", seed=7,
            constants={"greedy-compressed": {"seed": 7}},
            progbar=False, parallel=False)
    return path_optimizers[key]


def mps_memory():
    torch.mps.synchronize()
    gib = 2 ** 30
    return {
        "allocated_gib": torch.mps.current_allocated_memory() / gib,
        "driver_gib": torch.mps.driver_allocated_memory() / gib,
        "recommended_gib": torch.mps.recommended_max_memory() / gib,
    }


def model_arrays(cmap):
    return (
        cmap.boxes[0].channel.kraus.array,
        cmap.boxes[n_cells].channel.kraus.array,
    )


def completion_energy(model, clues, completion, cmap=None):
    global active_max_bond
    model["contractions"] += 1
    active_max_bond = model["max_bond"]
    cmap = sudoku_cmap(model) if cmap is None else cmap
    network = at_time_tensor_map(
        cmap, clues, completion, model["unroll_steps"])
    mantissa, exponent = contract_tensor(
        network, backend="pytorch",
        optimize=contraction_optimizer(
            model["unroll_steps"], model["max_bond"]),
        max_bond=model["max_bond"], cutoff=1e-8, dtype=tensor_dtype,
        canonize_distance=0, canonize_after_distance=0,
        compress_opts={"method": "fixed-rank-qr"},
        strip_exponent=True)
    log_magnitude = (
        torch.log(mantissa.array.abs().real.clamp_min(
            torch.finfo(tensor_dtype).tiny))
        + exponent * np.log(10))
    return -2 * log_magnitude


def correct_probabilities(model, dataset):
    probabilities = []
    for clues, solution, invalid in dataset:
        cmap = sudoku_cmap(model)
        energies = torch.stack((
            completion_energy(model, clues, solution, cmap),
            completion_energy(model, clues, invalid, cmap),
        ))
        probabilities.append(torch.softmax(-energies, 0)[0].item())
    return probabilities


def cell_distribution(model, clues, cell, cmap=None):
    cmap = sudoku_cmap(model) if cmap is None else cmap
    energies = []
    for digit in range(1, size + 1):
        completion = np.zeros(n_cells, dtype=int)
        completion[cell] = digit
        energies.append(completion_energy(
            model, clues, completion, cmap))
    return torch.softmax(-torch.stack(energies), 0)


def decode(model, dataset):
    n_correct, n_hidden, n_solved = 0, 0, 0
    grids = []
    for clues, solution, _ in dataset:
        prediction = clues.copy()
        cmap = sudoku_cmap(model)
        for cell in np.flatnonzero(clues == 0):
            probabilities = cell_distribution(model, clues, cell, cmap)
            prediction[cell] = probabilities.argmax().item() + 1
            n_correct += prediction[cell] == solution[cell]
            n_hidden += 1
        n_solved += np.array_equal(prediction, solution)
        grids.append(prediction.reshape(size, size).tolist())
    return {
        "per_cell_accuracy": n_correct / n_hidden,
        "full_grid_solve_rate": n_solved / len(dataset),
        "grids": grids,
    }


def targeted_accuracy(model, dataset):
    n_correct, predictions = 0, []
    for case_index, (clues, solution, _) in enumerate(dataset):
        hidden = np.flatnonzero(clues == 0)
        cell = hidden[case_index % len(hidden)]
        probabilities = cell_distribution(model, clues, cell)
        prediction = probabilities.argmax().item() + 1
        predictions.append(prediction)
        n_correct += prediction == solution[cell]
    return n_correct / len(dataset), predictions


pilot_updates, batch_size = 8, 2
scale_updates = 24
budget = {
    "probes": 48,
    "pilot_training_per_depth": (
        pilot_updates * batch_size * size),
    "pilot_candidate_evaluation_per_depth": (
        2 * 2 * len(validation_cases)),
    "pilot_readout_per_depth": (
        len(validation_cases) * size),
    "pilot_total_per_depth": 128,
    "three_pilots": 3 * 128,
    "winner_scale_training": (
        scale_updates * batch_size * size),
    "winner_candidate_evaluation": 2 * len(test_cases),
    "winner_readout": len(readout_cases) * 8 * size,
    "winner_scale_total": 448,
    "maximum_experiment_contractions": 880,
}
assert sum((
    budget["pilot_training_per_depth"],
    budget["pilot_candidate_evaluation_per_depth"],
    budget["pilot_readout_per_depth"],
)) == budget["pilot_total_per_depth"]
assert sum((
    budget["winner_scale_training"],
    budget["winner_candidate_evaluation"],
    budget["winner_readout"],
)) == budget["winner_scale_total"]
assert budget["probes"] + budget["three_pilots"] + (
    budget["winner_scale_total"]
) == budget["maximum_experiment_contractions"]
budget

{'pilot_training_per_ansatz': 192,
 'pilot_candidate_evaluation_per_ansatz': 32,
 'pilot_readout_per_ansatz': 256,
 'pilot_total_per_ansatz': 480,
 'three_pilots': 1440,
 'winner_scale_training': 576,
 'winner_candidate_evaluation': 128,
 'winner_readout': 512,
 'winner_scale_total': 1216,
 'lower_rate_restart_training': 768,
 'lower_rate_restart_initial_candidates': 16,
 'lower_rate_restart_final_candidates': 128,
 'lower_rate_restart_readout': 256,
 'lower_rate_restart_total': 1168,
 'maximum_experiment_contractions': 2656}

In [9]:
def four_way_loss(model, clues, solution, cell, cmap=None):
    cmap = sudoku_cmap(model) if cmap is None else cmap
    energies = []
    for digit in range(1, size + 1):
        completion = np.zeros(n_cells, dtype=int)
        completion[cell] = digit
        energies.append(completion_energy(
            model, clues, completion, cmap))
    energies = torch.stack(energies)
    target = solution[cell] - 1
    probabilities = torch.softmax(-energies, 0)
    alternatives = torch.cat((
        energies[:target], energies[target + 1:]))
    diagnostics = {
        "target_probability": probabilities[target].item(),
        "energy_span": (energies.max() - energies.min()).item(),
        "target_margin": (
            alternatives.min() - energies[target]).item(),
    }
    loss = energies[target] + torch.logsumexp(-energies, 0)
    return loss, diagnostics


def train_model(
        model, dataset, start_update, n_updates, optimizer=None,
        learning_rate=.003):
    parameters = model["cell"], model["constraint"]
    if optimizer is None:
        optimizer = torch.optim.Adam(
            parameters, lr=learning_rate)
    history = []
    diagnostics = {
        "target_probability": [], "energy_span": [],
        "target_margin": [], "gradient_norm": [],
    }
    for update in range(start_update, start_update + n_updates):
        optimizer.zero_grad()
        losses = []
        cmap = sudoku_cmap(model)
        arrays = model_arrays(cmap)
        accumulated = [torch.zeros_like(array) for array in arrays]
        for offset in range(batch_size):
            case_index = update * batch_size + offset
            clues, solution, _ = dataset[case_index % len(dataset)]
            hidden = np.flatnonzero(clues == 0)
            cell = hidden[case_index % len(hidden)]
            loss, example_diagnostics = four_way_loss(
                model, clues, solution, cell, cmap)
            gradients = torch.autograd.grad(loss / batch_size, arrays)
            for total, gradient in zip(accumulated, gradients):
                total.add_(gradient.detach())
            losses.append(loss.item())
            for name, value in example_diagnostics.items():
                diagnostics[name].append(value)
        torch.autograd.backward(arrays, accumulated)
        diagnostics["gradient_norm"].append(torch.sqrt(sum(
            parameter.grad.square().sum()
            for parameter in parameters)).item())
        torch.nn.utils.clip_grad_norm_(parameters, max_norm=10)
        optimizer.step()
        history.append(float(np.mean(losses)))
    return optimizer, history, diagnostics


def distribution_summary(values):
    values = np.asarray(values)
    return {
        "minimum": float(values.min()),
        "median": float(np.median(values)),
        "maximum": float(values.max()),
        "mean": float(values.mean()),
    }


def run_probe(depth, ticks, max_bond):
    gc.collect()
    torch.mps.empty_cache()
    model = initialise_model(
        depth, unroll_steps=ticks - 1, max_bond=max_bond)
    result = {
        "depth": depth, "ticks": ticks, "chi": max_bond,
        "parameters": model["cell"].numel()
        + model["constraint"].numel(),
    }
    started = time.perf_counter()
    try:
        cmap = sudoku_cmap(model)
        build_seconds = time.perf_counter() - started
        clues, solution, _ = train_cases[0]
        cell = np.flatnonzero(clues == 0)[0]
        loss, diagnostics = four_way_loss(
            model, clues, solution, cell, cmap)
        forward_memory = mps_memory()
        loss.backward()
        backward_memory = mps_memory()
        result.update({
            "status": "ok",
            "box_build_seconds": build_seconds,
            "seconds": time.perf_counter() - started,
            "contractions": model["contractions"],
            "loss": loss.item(),
            "gradient_norm": torch.sqrt(sum(
                parameter.grad.square().sum()
                for parameter in (
                    model["cell"], model["constraint"]
                ))).item(),
            "allocated_gib": max(
                forward_memory["allocated_gib"],
                backward_memory["allocated_gib"]),
            "driver_gib": max(
                forward_memory["driver_gib"],
                backward_memory["driver_gib"]),
        })
    except RuntimeError as error:
        result.update({
            "status": "error",
            "seconds": time.perf_counter() - started,
            "contractions": model["contractions"],
            "error": str(error).splitlines()[0],
        })
    return result


def probe_is_safe(result):
    return (result["status"] == "ok"
            and result["seconds"] <= 30
            and result["allocated_gib"] <= 6)


def run_pilot(depth, unroll_steps, max_bond):
    model = initialise_model(depth, unroll_steps, max_bond)
    started = time.perf_counter()
    with torch.no_grad():
        initial = correct_probabilities(model, validation_cases)
    optimizer, history, diagnostics = train_model(
        model, train_cases, 0, pilot_updates)
    with torch.no_grad():
        final = correct_probabilities(model, validation_cases)
        readout_accuracy, predictions = targeted_accuracy(
            model, validation_cases)
    result = {
        "depth": depth,
        "cell_parameters": model["cell"].numel(),
        "constraint_parameters": model["constraint"].numel(),
        "gradient_norm": distribution_summary(
            diagnostics["gradient_norm"]),
        "training_target_probability": distribution_summary(
            diagnostics["target_probability"]),
        "training_energy_span": distribution_summary(
            diagnostics["energy_span"]),
        "training_target_margin": distribution_summary(
            diagnostics["target_margin"]),
        "optimizer_updates": len(history),
        "training_examples": len(history) * batch_size,
        "first_four_loss": float(np.mean(history[:4])),
        "last_four_loss": float(np.mean(history[-4:])),
        "initial_candidate_probability": float(np.mean(initial)),
        "final_candidate_probability": float(np.mean(final)),
        "final_candidate_accuracy": float(
            np.mean(np.asarray(final) > .5)),
        "targeted_per_cell_accuracy": readout_accuracy,
        "targeted_predictions": predictions,
        "contractions": model["contractions"],
        "seconds": time.perf_counter() - started,
    }
    return model, optimizer, history, diagnostics, result


capacity_probes = []
for depth in (8, 16, 32):
    probe = run_probe(depth, ticks=2, max_bond=4)
    capacity_probes.append(probe)
    if not probe_is_safe(probe):
        break
probe_depth = capacity_probes[-1]["depth"]
limit_probes = []
stop_ticks = False
for ticks in (2, 3, 4):
    if stop_ticks:
        break
    for max_bond in (4, 8, 16):
        if (ticks, max_bond) == (2, 4):
            probe = capacity_probes[-1]
        else:
            probe = run_probe(probe_depth, ticks, max_bond)
        limit_probes.append(probe)
        if not probe_is_safe(probe):
            stop_ticks = max_bond == 4
            break
probe_contractions = 4 * (
    len(capacity_probes) + len(limit_probes) - 1)
assert probe_contractions <= budget["probes"]
projected_contractions = (
    budget["three_pilots"] + budget["winner_scale_total"]
)
eligible = [
    probe for probe in limit_probes
    if probe["ticks"] == 3 and probe_is_safe(probe)
    and probe["seconds"] * projected_contractions / 4 <= 15 * 60
]
assert eligible, "No three-tick configuration fits the GPU budget"
training_probe = max(eligible, key=lambda probe: probe["chi"])
training_configuration = {
    "unroll_steps": training_probe["ticks"] - 1,
    "ticks": training_probe["ticks"],
    "chi": training_probe["chi"],
    "projected_seconds": (
        training_probe["seconds"] * projected_contractions / 4),
}
probe_results = {
    "capacity": capacity_probes, "contraction": limit_probes,
    "training_configuration": training_configuration,
    "probe_contractions": probe_contractions,
}


pilot_states, pilot_results = {}, {}
for depth in (8, 16, 32):
    model, optimizer, history, diagnostics, result = run_pilot(
        depth, training_configuration["unroll_steps"],
        training_configuration["chi"])
    pilot_states[depth] = (
        model, optimizer, history, diagnostics)
    pilot_results[depth] = result
    assert result["gradient_norm"]["minimum"] > 1e-6
    assert result["contractions"] == (
        budget["pilot_total_per_depth"])
winner = max(pilot_results, key=lambda depth: (
    pilot_results[depth]["targeted_per_cell_accuracy"],
    pilot_results[depth]["final_candidate_probability"],
))
{"probes": probe_results, "pilots": pilot_results}


{'bipartite': {'cell_parameters': 34,
  'constraint_parameters': 32,
  'gradient_norm': {'minimum': 392.1979675292969,
   'median': 3588.21435546875,
   'maximum': 32118.564453125,
   'mean': 7565.52134958903},
  'training_target_probability': {'minimum': 1.561598850230439e-07,
   'median': 0.04759282432496548,
   'maximum': 0.9999418258666992,
   'mean': 0.2567656578118737},
  'training_energy_span': {'minimum': 2.7283172607421875,
   'median': 6.912357330322266,
   'maximum': 15.671806335449219,
   'mean': 7.831705093383789},
  'training_target_margin': {'minimum': -15.671806335449219,
   'median': -2.89666748046875,
   'maximum': 10.326400756835938,
   'mean': -3.2815731366475425},
  'optimizer_updates': 12,
  'training_examples': 48,
  'first_six_loss': 4.170366923014323,
  'last_six_loss': 4.497580846150716,
  'initial_candidate_probability': 0.3862941723188271,
  'final_candidate_probability': 0.4957367509645012,
  'final_candidate_accuracy': 0.5,
  'final_per_cell_accuracy': 0.1

In [10]:
(winner_model, winner_optimizer, winner_history,
 winner_diagnostics) = pilot_states[winner]
scale_started = time.perf_counter()
winner_optimizer, additional_history, additional_diagnostics = (
    train_model(
        winner_model, train_cases, pilot_updates, scale_updates,
        winner_optimizer))
history = winner_history + additional_history
diagnostics = {
    name: winner_diagnostics[name] + additional_diagnostics[name]
    for name in winner_diagnostics
}
with torch.no_grad():
    final_test = correct_probabilities(winner_model, test_cases)
    final_readout = decode(winner_model, readout_cases)
scale_seconds = time.perf_counter() - scale_started
learning_contractions = sum(
    state[0]["contractions"] for state in pilot_states.values())
total_contractions = probe_contractions + learning_contractions
results = {
    "winner_depth": winner,
    "parameters": (winner_model["cell"].numel()
                   + winner_model["constraint"].numel()),
    "ticks": training_configuration["ticks"],
    "max_bond": training_configuration["chi"],
    "learning_rate": .003,
    "optimizer_updates": len(history),
    "training_examples": len(history) * batch_size,
    "first_eight_loss": float(np.mean(history[:8])),
    "last_eight_loss": float(np.mean(history[-8:])),
    "training_gradient_norm": distribution_summary(
        diagnostics["gradient_norm"]),
    "training_target_probability": distribution_summary(
        diagnostics["target_probability"]),
    "training_energy_span": distribution_summary(
        diagnostics["energy_span"]),
    "training_target_margin": distribution_summary(
        diagnostics["target_margin"]),
    "initial_validation_candidate_probability": (
        pilot_results[winner]["initial_candidate_probability"]),
    "final_validation_candidate_probability": float(
        np.mean(final_test[:len(validation_cases)])),
    "held_out_candidate_puzzles": len(final_test),
    "final_candidate_probability": float(np.mean(final_test)),
    "final_candidate_accuracy": float(
        np.mean(np.asarray(final_test) > .5)),
    "held_out_readout_puzzles": len(readout_cases),
    "final_per_cell_accuracy": final_readout["per_cell_accuracy"],
    "final_full_grid_solve_rate": (
        final_readout["full_grid_solve_rate"]),
    "winner_contractions": winner_model["contractions"],
    "learning_contractions": learning_contractions,
    "probe_contractions": probe_contractions,
    "total_contractions": total_contractions,
    "scale_seconds": scale_seconds,
}
assert results["training_examples"] == 64
assert results["winner_contractions"] == 576
assert results["learning_contractions"] == 832
assert results["total_contractions"] <= (
    budget["maximum_experiment_contractions"])
example = {
    "clues": test_cases[0][0].reshape(size, size).tolist(),
    "correct": test_cases[0][1].reshape(size, size).tolist(),
    "invalid": test_cases[0][2].reshape(size, size).tolist(),
    "prediction": final_readout["grids"][0],
    "correct_probability": final_test[0],
}
{"results": results, "example": example}

{'results': {'winner': 'ring',
  'mode': 'continued_pilot',
  'learning_rate': 0.01,
  'optimizer_updates': 48,
  'training_examples': 192,
  'first_twelve_loss': 5.1088487307230634,
  'last_twelve_loss': 4.256265481313069,
  'training_gradient_norm': {'minimum': 88.75841522216797,
   'median': 3953.25634765625,
   'maximum': 227061.34375,
   'mean': 26137.90064472622},
  'training_target_probability': {'minimum': 4.659864855094398e-11,
   'median': 0.039804328233003616,
   'maximum': 0.9999760389328003,
   'mean': 0.24025571631132486},
  'training_energy_span': {'minimum': 1.1703872680664062,
   'median': 11.656963348388672,
   'maximum': 44.556419372558594,
   'mean': 12.655694590674507},
  'training_target_margin': {'minimum': -23.6556396484375,
   'median': -3.0238876342773438,
   'maximum': 10.855552673339844,
   'mean': -4.019758648342556},
  'initial_validation_candidate_probability': 0.5393737998092547,
  'final_validation_candidate_probability': 0.619497106759809,
  'held_out_

## Finite `at_time` semantics

One unrolling gives two recurrent ticks, enough for a message to travel from a cell to a constraint and back. We deliberately do not call `fix`: convergence is not established for the learned channels. The executable probe below uses the public `at_time(1)` API on a recurrent wire with an explicit memory state; the full Sudoku contraction above is its direct combinatorial-map representation.

In [11]:
wire = Box("wire", qubit, qubit ** 2, Diagram.id(qubit ** 3))
probe = CMap([wire], [((0, 1), (0, 2))])
process = (
    Ket(1) @ Diagram.id(probe.memory) >> probe.step
).feedback(
    dom=probe.dom[:0], cod=probe.cod, mem=probe.memory,
    state=Ket(0) @ Ket(0))
late = process.at_time(
    training_configuration["unroll_steps"]).eval(DiscopyBackend())
assert np.allclose(late.density_matrix, [[0, 0], [0, 1]])

The full 192/64 data split is practical because dataset storage is cheap and contractions are sampled serially. In the GPU-only pilots, bipartite, ring and all-pairs channels reached 18.8%, 29.7% and 20.3% validation cell accuracy. The ring therefore received the full budget. After one 192-example pass it assigned the correct completion 0.581 mean probability over all 64 held-out puzzles and ranked it first 59.4% of the time. Exact four-way decoding reached 26.6% hidden-cell accuracy on 16 held-out puzzles, just above the 25% random baseline, and solved 0/16 full grids. Its median training target probability was only 0.040, median target margin was −3.02 and unclipped gradient norms ranged from 88.8 to 227,061. The ring exposes a weak learned constraint signal, but neither a larger leakage-free dataset nor a denser ansatz is enough to solve a grid at two ticks and `chi=4`. Memory ablation, more unroll depths and larger bond dimensions remain explicit benchmark work.